# Which discharge practices matter: ranking candidate exposures with TMLE

## At a glance

Hospital teams chose which of five discharge practices each patient going home got. Which of them
change the chance that a patient stays home, with no readmission, for 30 days? The page estimates
one effect for each practice, each in its own fit, and then adjusts the p-values for testing five
practices at once.

Booking a follow-up visit before discharge would raise the share who stay home by 0.089 (95%
interval 0.062 to 0.117), and a pharmacist's medication review by 0.044 (0.017 to 0.071). Neither
teach-back education nor a home-health referral shows an effect; in these simulated data neither
has one.

The caveat that matters most is what each fit holds fixed. Ranked with the other four practices
held as they were, the navigation offer looks weak: 0.030, with an adjusted p-value of 0.0794.
Navigators book the follow-up visits, though, and holding the bookings fixed leaves out the visits
the offer brings. Asked what offering navigation would do, the answer is 0.059 (0.032 to 0.086),
near the true 0.0506.

## The question

The health plan wants to know which practices to make standard for every patient going home. For
each practice the quantity is the change in the share of patients who stay home 30 days if every
patient got that practice, compared with no patient getting it: an average treatment effect, one
for each candidate practice (a *candidate exposure*).

```text
  W ──▶ every practice, and W ──▶ Y

  navigation ──▶ follow-up booked ──▶ Y
  navigation ──────────────────────▶ Y
  medication review ───────────────▶ Y
  teach-back          (no arrow to Y)
  home-health referral (no arrow to Y)

  W  = discharge_risk, age, prior_utilization
  Y  = stayed_home (no readmission by day 30)
  assumed absent: an unrecorded cause U with U ──▶ a practice and U ──▶ Y
```

The three recorded characteristics in W change both who gets each practice and the outcome, so
every fit adjusts for them. Navigators book follow-up visits, so the offer changes the booking,
and part of its effect runs through the visit. The missing U is the key assumption, and it is one
assumption per practice: nothing unrecorded drove both that practice and staying home.

## The data

Every fit sets its seeds, so a rerun with the version printed below gives the same numbers.

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.2.8


`make_discharge_practices` draws 4,000 simulated discharges and the true effect of each practice,
which real data never give.

In [2]:
from cleverly.datasets import PRACTICES, make_discharge_practices

# With your own data, load your data frame here, such as frame = pd.read_csv(...).
# Real data have no `truth`, so drop every line that uses it.
frame, truth = make_discharge_practices(n=4_000, seed=0)
# For your own data, change these lists: the practices to rank, and the characteristics
# recorded before any of them.
candidates = list(PRACTICES)
covariates = ["discharge_risk", "age", "prior_utilization"]
print("rows and columns:", frame.shape)
print(list(frame.columns))
print()
true_effects = pd.DataFrame(
    {
        "share who get it": [truth[f"share[{name}]"] for name in candidates],
        "others held as they were": [truth[f"adjusted[{name}]"] for name in candidates],
        "booking follows the offer": [truth[f"total[{name}]"] for name in candidates],
    },
    index=candidates,
)
print("true effects on the share who stay home:")
print(true_effects.round(4))

rows and columns: (4000, 9)
['stayed_home', 'transition_navigation', 'followup_booked', 'medication_review', 'teach_back', 'home_health_referral', 'discharge_risk', 'age', 'prior_utilization']

true effects on the share who stay home:
                       share who get it  others held as they were  booking follows the offer
transition_navigation            0.4344                    0.0165                     0.0506
followup_booked                  0.4668                    0.0984                     0.0984
medication_review                0.4538                    0.0493                     0.0493
teach_back                       0.5242                    0.0000                     0.0000
home_health_referral             0.3073                    0.0000                     0.0000


Each row is one discharge: whether the patient stayed home (1 or 0), the five practices (1 where
the patient got it), and three characteristics measured before discharge, on a standardized scale.
Two columns of true effects differ only for navigation: 0.0165 with each patient's booking held as
it was, and 0.0506 when the booking follows the offer. Teach-back and the home-health referral
have no effect.

## Why a simple comparison misleads

Teams chose who got each practice, so patients who got one can differ before discharge from those
who did not.

In [3]:
simple = pd.DataFrame(
    {
        "simple difference": [
            frame.loc[frame[name] == 1, "stayed_home"].mean()
            - frame.loc[frame[name] == 0, "stayed_home"].mean()
            for name in candidates
        ],
        "true effect": [truth[f"total[{name}]"] for name in candidates],
        "mean discharge_risk, got it": [
            frame.loc[frame[name] == 1, "discharge_risk"].mean() for name in candidates
        ],
        "mean discharge_risk, did not": [
            frame.loc[frame[name] == 0, "discharge_risk"].mean() for name in candidates
        ],
    },
    index=candidates,
)
print(simple.round(3))

                       simple difference  true effect  mean discharge_risk, got it  mean discharge_risk, did not
transition_navigation             -0.028        0.051                        0.365                        -0.307
followup_booked                    0.037        0.098                        0.258                        -0.248
medication_review                  0.002        0.049                        0.003                        -0.023
teach_back                         0.011        0.000                       -0.104                         0.097
home_health_referral              -0.133        0.000                        0.580                        -0.282


Patients referred to home health stayed home less often (by 0.133), though the referral does
nothing, and navigation looks harmful (-0.028) though it helps. Teams referred and offered
navigation more often to patients with a higher `discharge_risk` (0.580 against -0.282 for the
referral), and those patients are readmitted more anyway. A fair comparison adjusts for the
characteristics.

## Set up the analysis

`variable_importance` takes the outcome, the candidate practices and the characteristics to adjust
for, and fits each practice in turn as the treatment. By default each fit also adjusts for the
other four practices. `variable_importance_plan` lists what each fit will adjust for, and counts
the model fits, before fitting anything. The function takes the columns directly and records no
study description.

In [4]:
from cleverly import variable_importance_plan
from cleverly.estimators import TMLE

# Left out, the learners default to a Super Learner, which costs 30 to 120 seconds per fit.
# This page passes logistic regressions; with flexible learners keep the folds.
estimator = TMLE(
    outcome_learner=LogisticRegression(max_iter=1000),
    treatment_learner=LogisticRegression(max_iter=1000),
    n_folds=5,
    random_state=0,
)
arguments = {
    "outcome": "stayed_home",  # For your own data, change the outcome name.
    "candidates": candidates,
    "covariates": covariates,
    "estimator": estimator,
}
planned = variable_importance_plan(frame, **arguments)
print("model fits this will run:", planned.total_fits)
for name in candidates:
    others = [other for other in candidates if other != name]
    print(f"{name} adjusts for: the three characteristics and {', '.join(others)}")

model fits this will run: 50
transition_navigation adjusts for: the three characteristics and followup_booked, medication_review, teach_back, home_health_referral
followup_booked adjusts for: the three characteristics and transition_navigation, medication_review, teach_back, home_health_referral
medication_review adjusts for: the three characteristics and transition_navigation, followup_booked, teach_back, home_health_referral
teach_back adjusts for: the three characteristics and transition_navigation, followup_booked, medication_review, home_health_referral
home_health_referral adjusts for: the three characteristics and transition_navigation, followup_booked, medication_review, teach_back


Five fits of ten models each (an outcome and a treatment model in each of five folds) make 50.
Each practice's answer is causal only if its own conditions hold:

- no unrecorded factor drove both that practice and staying home (this needs judgment);
- every kind of patient had some chance of getting the practice and of not getting it, called
  *overlap* (the data can check this, in part);
- each practice means the same thing for every patient, and one patient's practices do not change
  another's outcome (these need judgment).

Navigation's fit adjusts for the follow-up booking, which the offer causes, so it needs one more:
no unrecorded factor drove both whether a visit was booked and staying home (this needs judgment).

## Rank the practices

Each fit uses targeted maximum likelihood estimation (TMLE). It fits two models: an outcome model,
which predicts staying home from the practices and the characteristics, and a treatment model,
which predicts each patient's chance of getting the practice. A correction step then uses the
treatment model to remove the bias the outcome model leaves. Each patient's predictions come from
models trained on the other four folds of patients.

In [5]:
from cleverly import variable_importance

ranking = variable_importance(frame, **arguments)
# The summary gives each practice's estimate, standard error, 95% interval, p-value for a zero
# effect, and that p-value adjusted for testing five practices.
print(ranking.summary())
print()
table = ranking.to_frame().set_index("candidate")
table["true effect"] = [truth[f"adjusted[{name}]"] for name in table.index]
table["covers truth"] = table["ci_lower"].le(table["true effect"]) & table["ci_upper"].ge(
    table["true effect"]
)
print(table[["psi", "ci_lower", "ci_upper", "true effect", "covers truth"]].round(4))

                                  Variable importance
Question:       effect of each candidate    Treatment model:        logistic regression
                on stayed_home, one fit     Rows:                                 4,000
                per candidate               Candidates:                               5
Outcome:        stayed_home (binary)        Multiple testing:     Benjamini-Hochberg, 5
Method:         targeted maximum                                                  tests
                likelihood (TMLE)
Outcome model:  logistic regression
                       estimate  std. error           95% CI  p-value  adjusted p-value
---------------------------------------------------------------------------------------
followup_booked           0.089       0.014   [0.062, 0.117]  <0.0001           <0.0001
medication_review         0.044       0.014   [0.017, 0.071]   0.0014            0.0034
transition_navigation     0.030       0.015   [0.000, 0.059]   0.0477            0.0

Booking a follow-up visit comes first, with 0.089 (0.062 to 0.117), then the medication review,
0.044 (0.017 to 0.071). Each of the five intervals contains its true value. Teach-back and the
home-health referral sit near zero, with adjusted p-values of 0.7214.

The rows are sorted by adjusted p-value, so the first row is the clearest effect, not the largest:
compare the estimates and their intervals to judge size. The adjustment controls the *false
discovery rate*, the expected share of practices with no effect among those you call significant,
when the five tests are independent or move together. Note [2] warns that these fits, which share
their patients, do not guarantee that condition.

## When one practice leads to another

Navigation's row above answers "what if we changed the offer while each patient's booking stayed as
it was?". The plan's question is what offering navigation would do, bookings included. The second
run stops adjusting each fit for the other practices (`adjust_for_other_candidates=False`) and adds
the offer to the characteristics, so the booking's fit still adjusts for the offer that causes it.
A practice is never adjusted for in its own fit, so navigation's fit adjusts for the three
characteristics only.

In [6]:
# For your own data, list here any practice that comes before others and can cause them.
offer_first = variable_importance(
    frame,
    **{**arguments, "covariates": [*covariates, "transition_navigation"]},
    adjust_for_other_candidates=False,
)
second = offer_first.to_frame().set_index("candidate")
rows = []
for label, fitted, key in (
    ("others held", table, "adjusted"),
    ("booking follows", second, "total"),
):
    for name in ("transition_navigation", "followup_booked"):
        row = fitted.loc[name]
        rows.append(
            {
                "fit": label,
                "practice": name,
                "psi": row["psi"],
                "ci_lower": row["ci_lower"],
                "ci_upper": row["ci_upper"],
                "p_value_adjusted": row["p_value_adjusted"],
                "its true effect": truth[f"{key}[{name}]"],
            }
        )
print(pd.DataFrame(rows).set_index(["fit", "practice"]).round(4))

                                          psi  ci_lower  ci_upper  p_value_adjusted  its true effect
fit             practice                                                                            
others held     transition_navigation  0.0297    0.0003    0.0588            0.0794           0.0165
                followup_booked        0.0892    0.0615    0.1168            0.0000           0.0984
booking follows transition_navigation  0.0590    0.0321    0.0858            0.0000           0.0506
                followup_booked        0.0893    0.0617    0.1169            0.0000           0.0984


Navigation's answer rises from 0.030 to 0.059 (0.032 to 0.086), and the new interval contains the
true 0.0506 and excludes 0.0165, the value with bookings held. The booking's estimate barely moves
(0.0892 against 0.0893), because adjusting for the offer alone is enough for it.

Adjust each candidate for the others only when none of them causes another. Holding a later
practice fixed leaves out the part of an effect that runs through it, and can add bias when
something unrecorded drives that practice and the outcome. In this sample the first interval also
contains 0.0506; in the review's 400 repeated draws (Go deeper) it missed 0.0506 on 254.

## Check the fit

The overlap check reads, for each practice, the chances its fitted treatment model predicts. A
chance near 0 or 1 gives a patient a huge weight. The *effective sample size* says how many
patients the weighted group is worth. The combined check (`assess()`) runs the overlap, model and
sensitivity checks on every practice's fit and names any that need attention.

In [7]:
support = ranking.diagnostics.support()
rows = {}
for name, report in support.items():
    sizes = report.effective_sample_size
    rows[name] = {
        "smallest chance": report.propensity_quantiles["overall"][0.0],
        "largest chance": report.propensity_quantiles["overall"][1.0],
        "share below 0.05": report.tail_mass[0.05]["below"],
        "effective share, got it": sizes["treated"]["ratio"],
        "effective share, did not": sizes["control"]["ratio"],
    }
print(pd.DataFrame(rows).T.round(4))
print()
plain = {"omitted_confounding": "unmeasured confounding (the sensitivity bound)"}
attention = [plain.get(item.name, item.name) for item in ranking.assess().attention]
print("checks that need attention:", ", ".join(attention) or "none")

                       smallest chance  largest chance  share below 0.05  effective share, got it  effective share, did not
transition_navigation           0.0301          0.9431            0.0040                   0.6556                    0.7523
followup_booked                 0.1050          0.8983            0.0000                   0.7922                    0.8052
medication_review               0.0676          0.9058            0.0000                   0.8773                    0.9066
teach_back                      0.2263          0.8422            0.0000                   0.9688                    0.9584
home_health_referral            0.0070          0.9351            0.0448                   0.5399                    0.8338

checks that need attention: unmeasured confounding (the sensitivity bound)


The home-health referral has the thinnest overlap: its smallest chance is 0.007, 4.5% of patients
are below 0.05, and the referred patients are worth 54% of their number after weighting. These
come from fitted logistic models that see only the three recorded characteristics; with your own
data, ask staff whether some patients were almost never referred. The one check that needs
attention, unmeasured confounding, is the sensitivity bound read in the next section. The combined
report cannot run its simulated-confounder check here, because these fits carry no study set-up for
it.

## What if a confounder is missing?

How strong would an unrecorded confounder have to be to erase each effect? The robustness value
answers with one number from 0 to 1: the strength that, equal on the outcome side (the share of
the outcome's remaining variation it explains) and on the weight side (how much it changes the
patients' weights), would pull the estimate to zero. The benchmark measures how strong a recorded
characteristic is on each side, and the bound then asks what a confounder of that strength would
do. The table covers the three practices with an effect.

In [8]:
fits = {
    "followup_booked": (ranking, "followup_booked"),
    "medication_review": (ranking, "medication_review"),
    "navigation, booking follows": (offer_first, "transition_navigation"),
}
rows = {}
for label, (run, name) in fits.items():
    fitted = run.fits[name]
    robustness = fitted.sensitivity.robustness_value()
    # Measure each recorded characteristic's strength, and keep the one whose bound reaches
    # lowest: a confounder as strong as the strongest recorded one.
    worst = None
    for characteristic in covariates:
        strength = fitted.sensitivity.benchmark(covariates=[characteristic])
        bound = fitted.sensitivity.omitted_confounding(cf_y=strength.cf_y, cf_d=strength.cf_d)
        if worst is None or bound.lower < worst[2].lower:
            worst = (characteristic, strength, bound)
    characteristic, strength, bound = worst
    rows[label] = {
        "robustness value": robustness["rv"],
        "at the 95% limit": robustness["rva"],
        "strongest characteristic": characteristic,
        "its outcome side": strength.cf_y,
        "its weight side": strength.cf_d,
        "bound, low end": bound.lower,
        "bound, high end": bound.upper,
    }
print(pd.DataFrame(rows).T.to_string(float_format=lambda value: f"{value:.3f}"))

                            robustness value at the 95% limit strongest characteristic its outcome side its weight side bound, low end bound, high end
followup_booked                        0.094            0.071           discharge_risk            0.099           0.028          0.042           0.137
medication_review                      0.050            0.025        prior_utilization            0.019           0.067          0.012           0.076
navigation, booking follows            0.063            0.039           discharge_risk            0.108           0.195         -0.088           0.206


A confounder weaker than 0.094 on both sides cannot erase the booking's effect, one weaker than
0.050 cannot erase the review's, and one weaker than 0.063 cannot erase navigation's total effect.
For each practice the cell also finds the recorded characteristic whose strength moves the bound
furthest: a confounder as strong as it would leave the booking's effect between 0.042 and 0.137,
and the review's between 0.012 and 0.076. Navigation's could fall anywhere from -0.088 to 0.206,
because teams offered navigation largely by `discharge_risk` (weight side 0.195), so a confounder
like it could erase the offer's effect. Only people who know how teams chose could judge whether an
unrecorded factor is that strong.

## What we can and can't conclude

For the health plan: booking a follow-up visit before discharge would raise the share who stay
home 30 days by about 0.09 (95% interval 0.062 to 0.117), a medication review by about 0.04 (0.017
to 0.071), and offering navigation, with the visits navigators book, by about 0.06 (0.032 to
0.086). Teach-back and home-health referrals show no effect at this size.

Each answer is its own causal question with its own conditions, the main one being that nothing
unrecorded drove both that practice and staying home. The booking's effect survives a confounder
as strong as any recorded characteristic, and the review's too; navigation's would not survive one
as strong as discharge risk. The effects are not shares of one total and do not add up. The
adjusted p-values guard against false leads among the practices called significant, with a
condition these fits do not guarantee. The page shows one simulated draw with a known answer.

## Go deeper

### How many false leads the adjustment lets through

Testing five practices gives five chances for a practice with no effect to reach p below 0.05 by
luck. The Benjamini-Hochberg adjustment raises the p-values so that, among the practices called
significant, the expected share with no effect stays at or below the chosen level.

In [9]:
columns = ["p_value", "p_value_adjusted"]
print(table[columns].round(4))

                       p_value  p_value_adjusted
candidate                                       
followup_booked         0.0000            0.0000
medication_review       0.0014            0.0034
transition_navigation   0.0477            0.0794
teach_back              0.7214            0.7214
home_health_referral    0.6940            0.7214


Navigation's raw p-value of 0.0477 is below 0.05, and its adjusted p-value of 0.0794 is not. The
adjustment is not a correction for one row: it depends on the other p-values, so a row's adjusted
value changes when practices join or leave the list.

A simulation script kept with this page's review records
(`reviews/notebook-review/probes/ranking-exposures/sweep.py`) repeats both runs on 400 draws of the
same simulated data. Teach-back or the home-health referral reached a raw p-value below 0.05 on
33 of the draws, and an adjusted one on 20. The mean share of false leads among the practices
called significant was 0.0158 (standard error 0.0035), near the 0.02 the adjustment promises here (0.05 times
two no-effect practices out of five). That is one law, not a general check, and the adjustment
controls neither the chance of any false lead nor the share in one table. No simulation study yet
checks it for related fits like these
([X125](../roadmap.md#x125-registered-study-of-variable-importance)).

### Try to break the result

These checks change the data in a known way and refit each practice. Shuffling the practice among
patients (`placebo`) should leave an effect near zero. Adding a column of pure noise
(`random_common_cause`) should barely move the estimate. Refitting on random 70% samples (`subset`)
should give estimates that spread as much as the standard error predicts for that size.

In [10]:
refutation = ranking.diagnostics.refute(tests=("placebo", "random_common_cause", "subset"))
columns = ["candidate", "test", "original", "refuted_mean", "refuted_sd", "expectation", "verdict"]
print(refutation.to_frame()[columns].round(4).to_string(index=False))

            candidate                test  original  refuted_mean  refuted_sd        expectation     verdict
transition_navigation             placebo    0.0297       -0.0021      0.0153                ~ 0 descriptive
transition_navigation random_common_cause    0.0297        0.0299      0.0005          ~ 0.02969 descriptive
transition_navigation              subset    0.0297        0.0263      0.0090 scatter ~ 0.009774 descriptive
      followup_booked             placebo    0.0892        0.0061      0.0138                ~ 0 descriptive
      followup_booked random_common_cause    0.0892        0.0892      0.0004          ~ 0.08917 descriptive
      followup_booked              subset    0.0892        0.0872      0.0076 scatter ~ 0.009231 descriptive
    medication_review             placebo    0.0439       -0.0016      0.0094                ~ 0 descriptive
    medication_review random_common_cause    0.0439        0.0439      0.0004          ~ 0.04389 descriptive
    medication_revi

With each practice shuffled, the refits average close to zero, such as 0.0061 for the booking
against its estimate of 0.0892. The noise column moves no estimate by more than 0.0002, and the 70%
refits spread about as much as the `expectation` column predicts for a sample of that size. Every
`verdict` reads `descriptive`, no pass or fail: the simulation studies behind the verdicts use other
laws and models, so these numbers describe the draws and test nothing. None of these checks can
detect a missing confounder.

### How we know the method works

Each practice's fit is a cross-fitted TMLE of an average treatment effect on a yes-or-no outcome,
and each row prints without an `experimental` mark. The study pages hold the results.

| section | simulation study | its data and models |
| --- | --- | --- |
| Rank the practices | [stacked point-treatment CV-TMLE](../technical-reference/method-evidence/stacked-point-treatment-cv-tmle.md) | another law, with other learners |
| the interval of a difference of two proportions | [asymmetric Wald intervals](../technical-reference/method-evidence/asymmetric-wald-intervals.md) | other laws |
| What if a confounder is missing? | [omitted-variable bound standard error](../technical-reference/method-evidence/omitted-variable-bound-standard-error.md), for the limit at the 95% interval | a linear law with correct models fitted in sample |
| Try to break the result | [point-treatment refutation rules](../technical-reference/method-evidence/refuter-calibration-point.md) | other laws |
| How many false leads the adjustment lets through | none: no study checks the adjusted p-values over related fits | |

The script under `reviews/notebook-review/probes/` that the multiplicity lesson cites was run for
this page: it checks readings of the page, and it is not a simulation study that validates a
method. The [method reference](../technical-reference/validation-methods.md#variable-importance)
says what the procedure computes and what it refuses.

## Where to go next

[Rank candidate exposures](../user-guide/variable-importance.md) covers candidates with more than
two levels, ratio effects, and runs that release each fit to save memory. The
[point-treatment example](point-treatment-tmle.ipynb) runs every check on one treatment, and [gain
over current practice](population-impact.ipynb) estimates an effect with a later step held at one
level for everyone, the formal version of holding the booking fixed. The [examples index](index.md)
lists every example.